# 5. Parallelism

*Using Microsoft Semantic Kernel (Agent Framework)*

Shows how to execute multiple agent tasks in parallel for improved efficiency. Learn how to manage state across parallel executions and combine results from multiple concurrent operations.

In [ ]:
import os
import asyncio
from dotenv import load_dotenv
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import ChatHistory

load_dotenv()

kernel = sk.Kernel()
service_id = "chat-gpt"
kernel.add_service(
    AzureChatCompletion(
        service_id=service_id,
        deployment_name="gpt-4o-mini",
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    )
)

chat_service = kernel.get_service(service_id)

In [ ]:
async def process_query(query: str, agent_role: str) -> str:
    """Process a query with a specific agent role."""
    chat_history = ChatHistory()
    chat_history.add_system_message(f"You are a {agent_role}.")
    chat_history.add_user_message(query)
    
    response = await chat_service.get_chat_message_content(
        chat_history=chat_history,
        settings=kernel.get_prompt_execution_settings_from_service_id(service_id),
    )
    
    return str(response)

In [ ]:
async def parallel_analysis(topic: str):
    """Analyze a topic from multiple perspectives in parallel."""
    
    # Define different agents with different perspectives
    agents = [
        ("technical expert", f"Analyze {topic} from a technical perspective"),
        ("business analyst", f"Analyze {topic} from a business perspective"),
        ("user experience designer", f"Analyze {topic} from a UX perspective"),
    ]
    
    # Execute all queries in parallel
    tasks = [process_query(query, role) for role, query in agents]
    results = await asyncio.gather(*tasks)
    
    # Combine results
    combined = "\n\n".join([
        f"=== {agents[i][0].upper()} ==\n{result}"
        for i, result in enumerate(results)
    ])
    
    return combined

In [ ]:
# Test parallel execution
result = await parallel_analysis("AI-powered chatbots")
print(result)

**Note:** Using Python's asyncio.gather(), we can execute multiple LLM calls in parallel, significantly reducing total execution time when processing independent queries.